In [ ]:
from LSB_steg import LSB
from PVD_steg import PVD
from DCT_steg import DCT, DCT_jpeg
from conv_net import ConvNet
from huffman import MessageParser
from aux import Metrics
from phase_encoding import PhaseEncoding

from bitstring import BitArray
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
import jpegio as jio
import copy

# LSB

Citesc imaginea originala pentru a avea un reper de comparatie

In [ ]:
path = '/home/alexmiclea/Documents/Facultate/Licenta/images/lenna/lenna.png'

image_read = Image.open(path).convert('RGB')
lenna_original = np.array(image_read)

image_shape = lenna_original.shape

print(f'Dimensiunea imaginii: {image_shape[0]} {image_shape[1]}')

plt.imshow(lenna_original)

Voi citi un fișier pdf pe care să îl ascund în imaginea mea

In [ ]:
exec_path = '/home/alexmiclea/Documents/Facultate/Licenta/pdf/nlp.pdf'

with open(exec_path, 'rb') as exec_file:
    exec_bytes = exec_file.read()

Convertesc mesajul într-un BitArray

In [ ]:
exec_bits = BitArray(exec_bytes)
len_message = len(exec_bits)
print(f'Dimensiunea fisierului executabil in biti este: {len(exec_bits)}')

Initializez modelul LSB pentru Lenna (cu spatiu de embedding pe 2 biti)

In [ ]:
model = LSB(path, 1)

In [ ]:
capacity = model.get_embedding_capacity()
print(f'Capacitatea de a ascunde in imagine (biti) este: {capacity}')

In [ ]:
lenna_embed = model.embed_message(exec_bits)

In [ ]:
fig, axs = plt.subplots(1,2)
fig.set_figwidth(10)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[0].imshow(lenna_original)
axs[0].set_title('Imaginea originală')
axs[1].imshow(lenna_embed)
axs[1].set_title('Imaginea cu mesaj ascuns')
plt.show()

In [ ]:
fig, axs = plt.subplots(1,2)
fig.set_figwidth(10)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[0].imshow((lenna_original[:,:,0] & 1) * 255, cmap='gray')
axs[0].set_title('Imaginea originală')
axs[1].imshow((lenna_embed[:,:,0] & 1) * 255, cmap='gray')
axs[1].set_title('Imaginea cu mesaj ascuns')

In [ ]:
mse = Metrics.get_mse(lenna_original, lenna_embed)
psnr = Metrics.get_psnr(lenna_original, lenna_embed)
mssim = Metrics.get_mssim(lenna_original, lenna_embed)

print(f'MSE: {mse}')
print(f'PSNR: {psnr}')
print(f'MSSIM: {mssim}')

Desigur, daca imaginea este alesa corespunzator, un atac vizual de acest tip poate detecta faptul ca o imagine a fost modificata

In [ ]:
path = '/home/alexmiclea/Documents/Facultate/Licenta/images/astronaut/astronaut.png'
model = LSB(path, 1)
image_read = Image.open(path).convert('RGB')
astronaut_original = np.array(image_read)
plt.imshow(astronaut_original)

In [ ]:
astronaut_embed = model.embed_message(exec_bits)
extracted_message = model.extract_message(astronaut_embed)
extracted_message = extracted_message[:len_message]
extracted_message_bytes = bytes.fromhex(hex(int(extracted_message.bin, 2))[2:])

In [ ]:
fig, axs = plt.subplots(1,2)
fig.set_figwidth(10)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[0].imshow((astronaut_original[:,:,0] & 1) * 255, cmap='gray')
axs[0].set_title('Imaginea originală')
axs[1].imshow((astronaut_embed[:,:,0] & 1) * 255, cmap='gray')
axs[1].set_title('Imaginea cu mesaj ascuns')

In [ ]:
mse = Metrics.get_mse(astronaut_original, astronaut_embed)
psnr = Metrics.get_psnr(astronaut_original, astronaut_embed)
mssim = Metrics.get_mssim(astronaut_original, astronaut_embed)

print(f'MSE: {mse}')
print(f'PSNR: {psnr}')
print(f'MSSIM: {mssim}')

In [ ]:
file = '/home/alexmiclea/Documents/Facultate/Licenta/fisiere_extrase/nlp.pdf'

if not os.path.exists(file):
    open(file, 'wb').close()


with open(file, 'wb') as exec_file:
    exec_file.write(extracted_message_bytes)

os.chmod(file, 0o777)

# PVD


In [ ]:
deveselu_path = '/home/alexmiclea/Documents/Facultate/Licenta/images/deveselu/deveselu.png'

image_read = Image.open(deveselu_path).convert('RGB')
deveselu_original = np.array(image_read)

image_shape = deveselu_original.shape

print(f'Dimensiunea imaginii: {image_shape[0]} {image_shape[1]}')

plt.imshow(deveselu_original)

In [ ]:
model = LSB(deveselu_path, 1)

capacity = model.get_embedding_capacity()
print(f'Capacitatea de a ascunde in imagine (biti) este: {capacity}')

In [ ]:
model = PVD(deveselu_path)

capacity = model.get_embedding_capacity()
print(f'Capacitatea de a ascunde in imagine (biti) este: {capacity}')

In [ ]:
fmi_path = '/home/alexmiclea/Documents/Facultate/Licenta/images/fmi/fmi.jpg'

with open(fmi_path, 'rb') as exec_file:
    fmi_bytes = exec_file.read()

In [ ]:
fmi_size = len(fmi_bytes) * 8
print(f'Dimensiunea fisierului in biti este: {fmi_size}')

In [ ]:
deveselu_embed = model.get_pvd_with_embedded_message(fmi_bytes)

In [ ]:
fig, axs = plt.subplots(1,2)
fig.set_figwidth(10)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[0].imshow((deveselu_original[:,:,0] & 1) * 255, cmap='gray')
axs[0].set_title('Imaginea originală')
axs[1].imshow((deveselu_embed[:,:,0] & 1) * 255, cmap='gray')
axs[1].set_title('Imaginea cu mesaj ascuns')

In [ ]:
mse = Metrics.get_mse(deveselu_original, deveselu_embed)
psnr = Metrics.get_psnr(deveselu_original, deveselu_embed)
mssim = Metrics.get_mssim(deveselu_original, deveselu_embed)

print(f'MSE: {mse}')
print(f'PSNR: {psnr}')
print(f'MSSIM: {mssim}')

In [ ]:
fig, axs = plt.subplots(1,2)
fig.set_figwidth(10)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[0].imshow((deveselu_original[:,:,1] & 2) * 127, cmap='gray')
axs[0].set_title('Imaginea originală')
axs[1].imshow((deveselu_embed[:,:,1] & 2) * 127, cmap='gray')
axs[1].set_title('Imaginea cu mesaj ascuns')

In [ ]:
is_image_altered = Metrics.PVD_detect_if_image_is_manipulated(deveselu_embed)
print(f'Este {'adevarat' if is_image_altered else 'fals'} faptul ca imaginea a fost alterata.')

In [ ]:
is_image_altered_r = Metrics.PVD_detect_if_image_is_manipulated(deveselu_embed[:,:,0])
print(f'Este {'adevarat' if is_image_altered_r else 'fals'} faptul ca imaginea a fost alterata pe canalul R.')

is_image_altered_g = Metrics.PVD_detect_if_image_is_manipulated(deveselu_embed[:,:,1])
print(f'Este {'adevarat' if is_image_altered_g else 'fals'} faptul ca imaginea a fost alterata pe canalul G.')

is_image_altered_b = Metrics.PVD_detect_if_image_is_manipulated(deveselu_embed[:,:,2])
print(f'Este {'adevarat' if is_image_altered_b else 'fals'} faptul ca imaginea a fost alterata pe canalul B.')

In [ ]:
extracted_message = model.extract_message(deveselu_embed)
extracted_message = extracted_message[:fmi_size]
extracted_message_bytes = bytes.fromhex(hex(int(extracted_message.bin, 2))[2:])

In [ ]:
file = '/home/alexmiclea/Documents/Facultate/Licenta/fisiere_extrase/fmi.jpg'

if not os.path.exists(file):
    open(file, 'wb').close()

with open(file, 'wb') as exec_file:
    exec_file.write(extracted_message_bytes)

os.chmod(file, 0o777)

# DCT

In [ ]:
path = '/home/alexmiclea/Documents/Facultate/Licenta/images/baboon/baboon.png'

image_read = Image.open(path).convert('RGB')
baboon_original = np.array(image_read)

plt.imshow(baboon_original)

In [ ]:
model = DCT('/home/alexmiclea/Documents/Facultate/Licenta/images/baboon/baboon.png')
print(model.print_embedding_capacity())

In [ ]:
exec_path = '/home/alexmiclea/Documents/Facultate/Licenta/proj_code/test_executable/main'

with open(exec_path, 'rb') as exec_file:
    exec_bytes = exec_file.read()

In [ ]:
exec_bits = BitArray(exec_bytes)
len_message = len(exec_bits)
print(f'Dimensiunea fisierului executabil in biti este: {len_message}')

In [ ]:
message = MessageParser.create_message_bits(exec_bytes)
print(len(message))
message_bytes = message.tobytes()

In [ ]:
image, y_coefs = model.get_dct_compressed_image()
image_embed, y_coefs_embed = model.get_dct_with_embedded_message(message_bytes)

In [ ]:
fig, axs = plt.subplots(1,3)
fig.set_figwidth(15)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[2].set_axis_off()
axs[0].imshow(baboon_original)
axs[0].set_title('Imaginea originală')
axs[1].imshow(image)
axs[1].set_title('Imaginea comprimată DCT')
axs[2].imshow(image_embed)
axs[2].set_title('Imaginea cu mesaj ascuns')

In [ ]:
mse = Metrics.get_mse(baboon_original, image)
psnr = Metrics.get_psnr(baboon_original, image)
mssim = Metrics.get_mssim(baboon_original, image)

print(f'MSE: {mse}')
print(f'PSNR: {psnr}')
print(f'MSSIM: {mssim}')

In [ ]:
mse = Metrics.get_mse(image_embed, image)
psnr = Metrics.get_psnr(image_embed, image)
mssim = Metrics.get_mssim(image_embed, image)

print(f'MSE: {mse}')
print(f'PSNR: {psnr}')
print(f'MSSIM: {mssim}')

In [ ]:
mse = Metrics.get_mse(image_embed, image)
psnr = Metrics.get_psnr(image_embed, image)
mssim = Metrics.get_mssim(image_embed, image)

print(f'MSE: {mse}')
print(f'PSNR: {psnr}')
print(f'MSSIM: {mssim}')

In [ ]:
fig, axs = plt.subplots(1,2)
fig.set_figwidth(10)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[0].imshow((image[:,:,0] & 3) * 85, cmap = 'gray')
axs[0].set_title('Imaginea comprimată DCT')
axs[1].imshow((image_embed[:,:,0] & 3) * 85, cmap = 'gray')
axs[1].set_title('Imaginea cu mesaj ascuns')

In [ ]:
embed, true = Metrics.DCT_get_frequencies(y_coefs, y_coefs_embed)

true[32] = 0
embed[32] = 0

fig, axs = plt.subplots(1,2)
fig.set_figwidth(10)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[0].stem(true[:,0], true[:,1])
axs[0].set_title('Imaginea comprimată DCT')
axs[1].stem(embed[:,0], embed[:,1])
axs[1].set_title('Imaginea cu mesaj ascuns')

In [ ]:
plt.imshow(np.abs(image - image_embed) * 16)
plt.axis('off')

In [ ]:
embedding_space = model.get_message_bytes_from_encoded_y_channel(y_coefs_embed)

print(len(embedding_space))

In [ ]:
original_message = MessageParser.retrieve_message_bytes(embedding_space)

file = '/home/alexmiclea/Documents/Facultate/Licenta/fisiere_extrase/main_dct_mine'

if not os.path.exists(file):
    open(file, 'wb').close()

with open(file, 'wb') as exec_file:
    exec_file.write(original_message)

os.chmod(file, 0o777)

# DCT (varianta JPEG)

In [ ]:
path = '/home/alexmiclea/Documents/Facultate/Licenta/images/barbara/barbara.jpg'
embed_path = '/home/alexmiclea/Documents/Facultate/Licenta/images/barbara/barbara_embed.jpg'

image_read = Image.open(path).convert('RGB')
barbara_original = np.array(image_read)

plt.imshow(barbara_original)

In [ ]:
exec_path = '/home/alexmiclea/Documents/Facultate/Licenta/proj_code/test_executable/main'
with open(exec_path, 'rb') as exec_file:
    exec_bytes = exec_file.read()
print(len(exec_bytes) * 8)

In [ ]:
model = DCT_jpeg(path)
print(model.print_embedding_capacity())

In [ ]:
message = MessageParser.create_message_bits(exec_bytes)
print(type(message))
print(len(message))

In [ ]:
message_bytes = message.tobytes()
print(len(message_bytes))

In [ ]:
barbara_embed = model.get_dct_with_embedded_message(message_bytes, embed_path)

In [ ]:
fig, axs = plt.subplots(1,2)
fig.set_figwidth(10)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[0].imshow(barbara_original, cmap = 'gray')
axs[0].set_title('Imaginea originală')
axs[1].imshow(barbara_embed, cmap = 'gray')
axs[1].set_title('Imaginea cu mesaj ascuns')

In [ ]:
plt.figure(figsize=(8,6))
plt.axis('off')
plt.imshow(np.abs(barbara_original.astype(np.int32) - barbara_embed.astype(np.int32)) * 8)

In [ ]:
mse = Metrics.get_mse(barbara_original, barbara_embed)
psnr = Metrics.get_psnr(barbara_original, barbara_embed)
mssim = Metrics.get_mssim(barbara_original, barbara_embed)

print(f'MSE: {mse}')
print(f'PSNR: {psnr}')
print(f'MSSIM: {mssim}')

In [ ]:
clear_image_metadata = jio.read(path)
embed_image_metadata = jio.read(embed_path)

clear_coefs = copy.deepcopy(clear_image_metadata.coef_arrays[0])
embed_coefs = copy.deepcopy(embed_image_metadata.coef_arrays[0])

embed, true = Metrics.DCT_get_frequencies(clear_coefs, embed_coefs)

true[32] = 0
embed[32] = 0

fig, axs = plt.subplots(1,2)
fig.set_figwidth(10)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[0].stem(true[:,0], true[:,1])
axs[0].set_title('Imaginea JPEG inițială')
axs[1].stem(embed[:,0], embed[:,1])
axs[1].set_title('Imaginea cu mesaj ascuns')

In [ ]:
embedding_space = model.get_message_bytes(embed_path)
print(len(embedding_space))

In [ ]:
original_message = MessageParser.retrieve_message_bytes(embedding_space)

file = '/home/alexmiclea/Documents/Facultate/Licenta/fisiere_extrase/main_dct_jpeg'

if not os.path.exists(file):
    open(file, 'wb').close()

with open(file, 'wb') as exec_file:
    exec_file.write(original_message)

os.chmod(file, 0o777)

In [ ]:
clean_path = '/home/alexmiclea/Documents/Facultate/Licenta/dataset/alaska2_coef_bincount/Cover/data.npy'
stego_path = '/home/alexmiclea/Documents/Facultate/Licenta/dataset/alaska2_coef_bincount/Cover_DCT_jpeg/data.npy'

test = ConvNet()
test.read_dataset(clean_path, stego_path)
test.load_model_weights('/home/alexmiclea/Documents/Facultate/Licenta/weights/weights_dct_jpeg.pth')
_, predictions, labels = test.eval(test.test_dataloader)
test.plot_confusion_matrix(predictions, labels, False)

# Phase Encoding

In [ ]:
model = PhaseEncoding('/home/alexmiclea/Documents/Facultate/Licenta/audio/numbers/numbers.wav', 1024)

In [ ]:
text_message = 'Acesta este un mesaj ascuns'
message_bytes = text_message.encode()
message = BitArray(bytes = message_bytes)

In [ ]:
model.embed_message(message, '/home/alexmiclea/Documents/Facultate/Licenta/audio/numbers/numbers_embed.wav')
message_extract = model.extract_message('/home/alexmiclea/Documents/Facultate/Licenta/audio/numbers/numbers_embed.wav')
message_extract = message_extract[:len(message)]

In [ ]:
extracted_message_bytes = message_extract.bytes
extracted_text_message = extracted_message_bytes.decode()
print(extracted_text_message)

In [ ]:
model = PhaseEncoding('/home/alexmiclea/Documents/Facultate/Licenta/audio/numbers/numbers.wav', 64)

In [ ]:
text_message = 'Acesta este un alt mesaj ascuns'
message_bytes = text_message.encode()
message = BitArray(bytes = message_bytes)

In [ ]:
model.embed_message(message, '/home/alexmiclea/Documents/Facultate/Licenta/audio/numbers/numbers_embed.wav')
message_extract = model.extract_message('/home/alexmiclea/Documents/Facultate/Licenta/audio/numbers/numbers_embed.wav')
message_extract = message_extract[:len(message)]

In [ ]:
extracted_message_bytes = message_extract.bytes
extracted_text_message = extracted_message_bytes.decode()
print(extracted_text_message)